# Treinamento e teste de diferentes modelos 

## Importando Dados

In [ ]:
from pathlib import Path

import joblib

raiz = Path.cwd()
if not (raiz / "Modelos").is_dir():
    raiz = raiz.parent

dados = joblib.load(raiz / "Modelos" / "dados_ml.joblib")

X_treino = dados["X"]["treino"]
y_treino = dados["y"]["treino"]

X_validacao = dados["X"]["validacao"]
y_validacao = dados["y"]["validacao"]

print("Treino:", X_treino.shape, y_treino.shape)
print("Validação:", X_validacao.shape, y_validacao.shape)

In [ ]:
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_treino, y_treino)

previsoes = baseline.predict(X_validacao)
scores = baseline.predict_proba(X_validacao)[:, 1]

print(f"Accuracy: {accuracy_score(y_validacao, previsoes):.2%}")
print(f"Precision: {precision_score(y_validacao, previsoes, zero_division=0):.2%}")
print(f"Recall: {recall_score(y_validacao, previsoes, zero_division=0):.2%}")
print(f"F1: {f1_score(y_validacao, previsoes, zero_division=0):.2%}")
print(f"Average Precision: {average_precision_score(y_validacao, scores):.4f}")

# Treinamento 1: Regressão Logística

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo_logistico = LogisticRegression(
    class_weight="balanced",
    max_iter=2000
)

modelo_logistico.fit(X_treino, y_treino)

print("Modelo treinado.")

In [ ]:
previsoes_log = modelo_logistico.predict(X_validacao)
scores_log = modelo_logistico.predict_proba(X_validacao)[:, 1]

print(f"Accuracy: {accuracy_score(y_validacao, previsoes_log):.2%}")
print(f"Precision: {precision_score(y_validacao, previsoes_log, zero_division=0):.2%}")
print(f"Recall: {recall_score(y_validacao, previsoes_log, zero_division=0):.2%}")
print(f"F1: {f1_score(y_validacao, previsoes_log, zero_division=0):.2%}")
print(f"Average Precision: {average_precision_score(y_validacao, scores_log):.4f}")

In [ ]:
7import pandas as pd

resultados_limites = []

for limite in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    alertas = (scores_log >= limite).astype(int)

    resultados_limites.append({
        "limite": limite,
        "precision_%": 100 * precision_score(y_validacao, alertas, zero_division=0),
        "recall_%": 100 * recall_score(y_validacao, alertas, zero_division=0),
        "f1_%": 100 * f1_score(y_validacao, alertas, zero_division=0),
        "pedidos_alertados": int(alertas.sum()),
        "falsos_alertas": int(((alertas == 1) & (y_validacao == 0)).sum())
    })

tabela_limites = pd.DataFrame(resultados_limites)
tabela_limites.round(2)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

limite_provisorio = 0.6
previsoes_validacao = (scores_log >= limite_provisorio).astype(int)

ConfusionMatrixDisplay.from_predictions(
    y_validacao,
    previsoes_validacao,
    display_labels=["No prazo", "Atrasou"],
    cmap="Blues",
    values_format="d"
)

plt.title("Regressão Logística — validação — limite 0.6")
plt.show()

# Avaliação no teste — Regressão logística

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo_logistico = LogisticRegression(
    class_weight="balanced",
    max_iter=2000
)

modelo_logistico.fit(
    dados["X"]["treino"],
    dados["y"]["treino"]
)

scores_log_teste = modelo_logistico.predict_proba(
    dados["X"]["teste"]
)[:, 1]

In [ ]:
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

avaliacao_log_teste = pd.DataFrame({
    "atrasou": dados["y"]["teste"],
    "score": scores_log_teste
})

quantidade_log = int(len(avaliacao_log_teste) * 0.10)
indices_log = avaliacao_log_teste.nlargest(
    quantidade_log, "score"
).index

avaliacao_log_teste["prioridade"] = 0
avaliacao_log_teste.loc[indices_log, "prioridade"] = 1

real_log = avaliacao_log_teste["atrasou"]
alertas_log = avaliacao_log_teste["prioridade"]

print(f"Pedidos avaliados: {len(avaliacao_log_teste)}")
print(f"Pedidos na fila: {quantidade_log}")
print(f"Atrasos encontrados: {int(real_log[alertas_log == 1].sum())}")
print(f"Precision: {precision_score(real_log, alertas_log):.2%}")
print(f"Recall: {recall_score(real_log, alertas_log):.2%}")
print(f"F1: {f1_score(real_log, alertas_log):.2%}")
print(f"AP: {average_precision_score(real_log, scores_log_teste):.4f}")
print("Matriz de confusão:")
print(confusion_matrix(real_log, alertas_log, labels=[0, 1]))

# Treinamento 2: HistGradientBoosting (HGB)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

modelo_hgb = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    class_weight="balanced",
    early_stopping=False,
    random_state=42
)

modelo_hgb.fit(X_treino, y_treino)
print("HGB treinado.")

In [ ]:
previsoes_hgb = modelo_hgb.predict(X_validacao)
scores_hgb = modelo_hgb.predict_proba(X_validacao)[:, 1]

print(f"Accuracy: {accuracy_score(y_validacao, previsoes_hgb):.2%}")
print(f"Precision: {precision_score(y_validacao, previsoes_hgb, zero_division=0):.2%}")
print(f"Recall: {recall_score(y_validacao, previsoes_hgb, zero_division=0):.2%}")
print(f"F1: {f1_score(y_validacao, previsoes_hgb, zero_division=0):.2%}")
print(f"Average Precision: {average_precision_score(y_validacao, scores_hgb):.4f}")

# Treinamento 3: Rede Neural (MLP)

In [ ]:
import numpy as np
import torch
from torch import nn

# Tensores são o formato de dados utilizado pelo PyTorch.
X_treino_tensor = torch.tensor(np.asarray(X_treino), dtype=torch.float32)
y_treino_tensor = torch.tensor(np.asarray(y_treino), dtype=torch.float32)

X_validacao_tensor = torch.tensor(np.asarray(X_validacao), dtype=torch.float32)
y_validacao_tensor = torch.tensor(np.asarray(y_validacao), dtype=torch.float32)

print("PyTorch:", torch.__version__)
print("Treino:", X_treino_tensor.shape, y_treino_tensor.shape)
print("Validação:", X_validacao_tensor.shape, y_validacao_tensor.shape)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from copy import deepcopy
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

# Configurações.
SEMENTE_CE = 7
EPOCAS_CE = 150
PACIENCIA_CE = 50
torch.manual_seed(SEMENTE_CE)

# Dados: somente treino e validação.
X_treino_ce = torch.tensor(np.asarray(X_treino), dtype=torch.float32)
y_treino_ce = torch.tensor(np.asarray(y_treino).ravel(), dtype=torch.long)
X_validacao_ce = torch.tensor(np.asarray(X_validacao), dtype=torch.float32)
y_validacao_ce = np.asarray(y_validacao).ravel().astype(int)
y_validacao_tensor_ce = torch.tensor(y_validacao_ce, dtype=torch.long)

# Rede nova, separada do modelo da aplicação.
modelo_ce = nn.Sequential(
    nn.Linear(X_treino_ce.shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)

# Pesos calculados exclusivamente no treino.
peso_atraso_ce = (
    (y_treino_ce == 0).sum() / (y_treino_ce == 1).sum()
).item()

criterio_ce = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, peso_atraso_ce], dtype=torch.float32)
)

otimizador_ce = torch.optim.AdamW(
    modelo_ce.parameters(), lr=0.0003, weight_decay=0.01
)

lotes_ce = DataLoader(
    TensorDataset(X_treino_ce, y_treino_ce),
    batch_size=256,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEMENTE_CE)
)

tamanho_fila_ce = int(len(y_validacao_ce) * 0.10)
melhor_quantidade_ce = -1
sem_melhora_ce = 0
historico_ce = []

for epoca_ce in range(1, EPOCAS_CE + 1):
    modelo_ce.train()

    for entradas_ce, alvos_ce in lotes_ce:
        otimizador_ce.zero_grad()
        logits_ce = modelo_ce(entradas_ce)
        perda_ce = criterio_ce(logits_ce, alvos_ce)
        perda_ce.backward()
        otimizador_ce.step()

    # Mede ambas as perdas ao final da época, com dropout desligado.
    modelo_ce.eval()
    with torch.no_grad():
        perda_treino_ce = criterio_ce(
            modelo_ce(X_treino_ce), y_treino_ce
        ).item()

        logits_validacao_ce = modelo_ce(X_validacao_ce)
        perda_validacao_ce = criterio_ce(
            logits_validacao_ce, y_validacao_tensor_ce
        ).item()

        scores_ce = torch.softmax(
            logits_validacao_ce, dim=1
        )[:, 1].numpy()

    indices_ce = np.argsort(
        -scores_ce, kind="stable"
    )[:tamanho_fila_ce]

    encontrados_ce = int(y_validacao_ce[indices_ce].sum())
    ap_ce = average_precision_score(y_validacao_ce, scores_ce)

    historico_ce.append({
        "epoca": epoca_ce,
        "perda_treino": perda_treino_ce,
        "perda_validacao": perda_validacao_ce,
        "average_precision": ap_ce,
        "atrasos_top10": encontrados_ce
    })

    print(
        f"Época {epoca_ce:03d} | "
        f"Perda treino: {perda_treino_ce:.4f} | "
        f"Perda validação: {perda_validacao_ce:.4f} | "
        f"AP: {ap_ce:.4f} | Atrasos na fila: {encontrados_ce}"
    )

    # Seleção e paciência continuam ligadas aos atrasos na fila.
    if encontrados_ce > melhor_quantidade_ce:
        melhor_quantidade_ce = encontrados_ce
        melhor_epoca_ce = epoca_ce
        melhor_estado_ce = deepcopy(modelo_ce.state_dict())
        melhores_scores_ce = scores_ce.copy()
        sem_melhora_ce = 0
    else:
        sem_melhora_ce += 1

    if sem_melhora_ce >= PACIENCIA_CE:
        print("Parada antecipada.")
        break

# Recupera a melhor época.
modelo_ce.load_state_dict(melhor_estado_ce)
modelo_ce.eval()

alertas_ce = np.zeros(len(y_validacao_ce), dtype=int)
indices_ce = np.argsort(
    -melhores_scores_ce, kind="stable"
)[:tamanho_fila_ce]
alertas_ce[indices_ce] = 1

print(f"\nMelhor época: {melhor_epoca_ce}")
print(f"Atrasos encontrados: {melhor_quantidade_ce} em {tamanho_fila_ce}")
print(f"Precision: {precision_score(y_validacao_ce, alertas_ce, zero_division=0):.2%}")
print(f"Recall: {recall_score(y_validacao_ce, alertas_ce, zero_division=0):.2%}")
print(f"F1: {f1_score(y_validacao_ce, alertas_ce, zero_division=0):.2%}")
print(f"AP: {average_precision_score(y_validacao_ce, melhores_scores_ce):.4f}")
print("Matriz de confusão:")
print(confusion_matrix(y_validacao_ce, alertas_ce, labels=[0, 1]))

# Gráficos.
epocas_grafico_ce = [r["epoca"] for r in historico_ce]
perdas_treino_ce = [r["perda_treino"] for r in historico_ce]
perdas_validacao_ce = [r["perda_validacao"] for r in historico_ce]
atrasos_fila_ce = [r["atrasos_top10"] for r in historico_ce]

fig_ce, eixos_ce = plt.subplots(
    1, 2, figsize=(13, 4.5), layout="constrained"
)

eixos_ce[0].plot(
    epocas_grafico_ce, perdas_treino_ce,
    label="Treino", color="#2563eb"
)
eixos_ce[0].plot(
    epocas_grafico_ce, perdas_validacao_ce,
    label="Validação", color="#d97706"
)
eixos_ce[0].set_title("Convergência — CrossEntropyLoss")
eixos_ce[0].set_ylabel("Perda ponderada — menor é melhor")

eixos_ce[1].plot(
    epocas_grafico_ce, atrasos_fila_ce,
    label="Atrasos encontrados", color="#15803d"
)
eixos_ce[1].scatter(
    melhor_epoca_ce, melhor_quantidade_ce,
    color="#dc2626", s=55, zorder=3
)
eixos_ce[1].set_title("Desempenho na validação")
eixos_ce[1].set_ylabel("Atrasos encontrados na fila de 10%")

for eixo_ce in eixos_ce:
    eixo_ce.axvline(
        melhor_epoca_ce,
        color="#555555",
        linestyle="--",
        label=f"Época selecionada: {melhor_epoca_ce}"
    )
    eixo_ce.set_xlabel("Época")
    eixo_ce.xaxis.set_major_locator(MaxNLocator(integer=True))
    eixo_ce.grid(alpha=0.2)
    eixo_ce.legend()

plt.show()

In [ ]:
from torch import nn

torch.manual_seed(7)

modelo_rede = nn.Sequential(
    nn.Linear(X_treino_tensor.shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

print(modelo_rede)

In [ ]:
# Calculamos o desbalanceamento usando somente o treino.
quantidade_atrasos = (y_treino_tensor == 1).sum()
quantidade_pontuais = (y_treino_tensor == 0).sum()
peso_atraso = quantidade_pontuais / quantidade_atrasos

criterio = nn.BCEWithLogitsLoss(
    pos_weight=peso_atraso.reshape(1)
)

otimizador = torch.optim.AdamW(
    modelo_rede.parameters(),
    lr=0.0003,
    weight_decay=0.01
)

print(f"Peso dos atrasos: {peso_atraso.item():.2f}")

In [ ]:
import pandas as pd
from copy import deepcopy
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score

torch.manual_seed(7)

dataset_treino = TensorDataset(X_treino_tensor, y_treino_tensor)

lotes_treino = DataLoader(
    dataset_treino,
    batch_size=256,
    shuffle=True,
    generator=torch.Generator().manual_seed(7)
)

In [ ]:
historico_rede = []
melhor_quantidade = -1
tamanho_fila = int(len(y_validacao_tensor) * 0.10)

paciencia = 50
epocas_sem_melhora = 0

for epoca in range(1, 151):
    # Aprendizado: somente dados de treino.
    modelo_rede.train()
    soma_erros = 0.0

    for entradas, alvos in lotes_treino:
        otimizador.zero_grad()

        saidas = modelo_rede(entradas).squeeze(1)
        erro = criterio(saidas, alvos)

        erro.backward()
        otimizador.step()

        soma_erros += erro.item() * len(entradas)

    erro_medio = soma_erros / len(dataset_treino)

    # Validação: calcula resultados sem ajustar os parâmetros.
    modelo_rede.eval()
    with torch.no_grad():
        saidas_validacao = modelo_rede(X_validacao_tensor).squeeze(1)
        scores = torch.sigmoid(saidas_validacao).numpy()

    avaliacao = pd.DataFrame({
        "atrasou": y_validacao_tensor.numpy(),
        "score": scores
    })

    fila = avaliacao.nlargest(tamanho_fila, "score")
    encontrados = int(fila["atrasou"].sum())
    ap = average_precision_score(avaliacao["atrasou"], scores)

    historico_rede.append({
        "epoca": epoca,
        "erro_treino": erro_medio,
        "average_precision": ap,
        "atrasos_top10": encontrados
    })

    # Guarda a melhor versão e acompanha épocas sem melhoria.
    if encontrados > melhor_quantidade:
        melhor_quantidade = encontrados
        melhor_epoca = epoca
        melhor_estado = deepcopy(modelo_rede.state_dict())
        scores_rede = scores.copy()
        epocas_sem_melhora = 0
    else:
        epocas_sem_melhora += 1

    print(
        f"Época {epoca:02d} | Erro: {erro_medio:.4f} | "
        f"AP: {ap:.4f} | Atrasos na fila: {encontrados}"
    )

    if epocas_sem_melhora >= paciencia:
        print(f"\nParada antecipada na época {epoca}.")
        break

# Recupera os parâmetros da melhor época.
modelo_rede.load_state_dict(melhor_estado)
modelo_rede.eval()

print(f"\nMelhor época: {melhor_epoca}")
print(f"Atrasos encontrados: {melhor_quantidade} em {tamanho_fila} pedidos")

# Métricas Rede Neural 

In [ ]:
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score
)

avaliacao_rede = pd.DataFrame({
    "atrasou": y_validacao_tensor.numpy().astype(int),
    "score": scores_rede
})

# Marca os 10% dos pedidos com maior score.
quantidade = int(len(avaliacao_rede) * 0.10)
indices = avaliacao_rede.nlargest(quantidade, "score").index

avaliacao_rede["prioridade"] = 0
avaliacao_rede.loc[indices, "prioridade"] = 1

real = avaliacao_rede["atrasou"]
alertas = avaliacao_rede["prioridade"]

print(f"Precision: {precision_score(real, alertas):.2%}")
print(f"Recall: {recall_score(real, alertas):.2%}")
print(f"F1: {f1_score(real, alertas):.2%}")
print(f"AP: {average_precision_score(real, scores_rede):.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

grafico = ConfusionMatrixDisplay.from_predictions(
    real,
    alertas,
    labels=[0, 1],
    display_labels=["No prazo", "Atrasou"],
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

grafico.ax_.set_xticklabels(["Fora da fila", "Na fila"])
grafico.ax_.set_xlabel("Priorização do modelo")
grafico.ax_.set_ylabel("Resultado observado")
grafico.ax_.set_title(
    "Rede neural — validação — fila de 10%\nSemente 7"
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    PrecisionRecallDisplay, precision_score, recall_score
)

real = avaliacao_rede["atrasou"]
alertas = avaliacao_rede["prioridade"]

curva = PrecisionRecallDisplay.from_predictions(
    real,
    avaliacao_rede["score"],
    name="Rede neural — semente 7"
)

curva.ax_.axhline(
    real.mean(),
    color="gray",
    linestyle="--",
    label="Referência: proporção de atrasos"
)

curva.ax_.scatter(
    recall_score(real, alertas),
    precision_score(real, alertas),
    color="red",
    label="Fila de 10%",
    zorder=3
)

curva.ax_.set_xlabel("Recall — parcela dos atrasos encontrados")
curva.ax_.set_ylabel("Precision — acerto entre os alertas")
curva.ax_.set_title("Precision–Recall — validação")
curva.ax_.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

modelo_rede.eval()

with torch.no_grad():
    scores_conferencia = torch.sigmoid(
        modelo_rede(X_validacao_tensor).squeeze(1)
    ).numpy()

np.testing.assert_allclose(
    scores_conferencia,
    avaliacao_rede["score"].to_numpy(),
    rtol=1e-5,
    atol=1e-7
)

print("Modelo corresponde aos resultados apresentados.")

In [ ]:
from sklearn.metrics import confusion_matrix

X_teste_tensor = torch.tensor(
    np.asarray(dados["X"]["teste"]),
    dtype=torch.float32
)

modelo_rede.eval()

with torch.no_grad():
    scores_teste = torch.sigmoid(
        modelo_rede(X_teste_tensor).squeeze(1)
    ).numpy()

avaliacao_teste = pd.DataFrame({
    "atrasou": np.asarray(dados["y"]["teste"]),
    "score": scores_teste
})

quantidade_teste = int(len(avaliacao_teste) * 0.10)
indices = avaliacao_teste.nlargest(quantidade_teste, "score").index

avaliacao_teste["prioridade"] = 0
avaliacao_teste.loc[indices, "prioridade"] = 1

real_teste = avaliacao_teste["atrasou"]
alertas_teste = avaliacao_teste["prioridade"]

print(f"Pedidos avaliados: {len(avaliacao_teste)}")
print(f"Pedidos na fila: {quantidade_teste}")
print(f"Frequência de atraso: {real_teste.mean():.2%}")
print(f"Precision: {precision_score(real_teste, alertas_teste):.2%}")
print(f"Recall: {recall_score(real_teste, alertas_teste):.2%}")
print(f"F1: {f1_score(real_teste, alertas_teste):.2%}")
print(f"AP: {average_precision_score(real_teste, scores_teste):.4f}")
print("Matriz de confusão:")
print(confusion_matrix(real_teste, alertas_teste, labels=[0, 1]))

# Comparação e priorização

In [ ]:
resultados_fila = []

for nome, scores in {"Logística": scores_log, "HGB": scores_hgb}.items():
    avaliacao = pd.DataFrame({
        "atrasou": y_validacao,
        "score": scores
    })

    for percentual in [5, 10, 20]:
        quantidade = int(len(avaliacao) * percentual / 100)
        selecionados = avaliacao.nlargest(quantidade, "score")
        atrasos = int(selecionados["atrasou"].sum())

        resultados_fila.append({
            "modelo": nome,
            "revisao_%": percentual,
            "pedidos_revisados": quantidade,
            "atrasos_encontrados": atrasos,
            "precision_%": 100 * atrasos / quantidade,
            "recall_%": 100 * atrasos / avaliacao["atrasou"].sum()
        })

pd.DataFrame(resultados_fila).round(2)

## Comparação final no teste

| Indicador | Regressão logística | Rede neural — semente 7 |
|---|---:|---:|
| Pedidos revisados | 2.555 | 2.555 |
| Atrasos encontrados | 276 | 285 |
| Precision | 10,80% | 11,15% |
| Recall | 24,56% | 25,36% |
| F1 | 15,00% | 15,49% |
| AP | 0,0885 | 0,0938 |
| Alertas sobre pedidos pontuais | 2.279 | 2.270 |

## Salvamento do modelo e do pré-processamento

In [ ]:
import joblib
import torch
import sklearn

pasta_modelos = raiz / "Modelos"

# Coloca a rede em modo de previsão.
modelo_rede.eval()

# Salva os pesos aprendidos pela rede.
torch.save(
    modelo_rede.state_dict(),
    pasta_modelos / "rede_neural.pt"
)

# Guarda o tratamento aprendido no treino e as configurações.
pacote_inferencia = {
    "preprocessador": dados["preprocessador"],
    "colunas_entrada": dados["colunas_entrada"],
    "nomes_atributos": dados["nomes_atributos"],
    "numero_entradas": modelo_rede[0].in_features,
    "camadas_ocultas": [
        modelo_rede[0].out_features,
        modelo_rede[3].out_features
    ],
    "dropout": modelo_rede[2].p,
    "semente": 7,
    "melhor_epoca": int(melhor_epoca),
    "fracao_fila": 0.10,
    "versao_torch": str(torch.__version__),
    "versao_sklearn": sklearn.__version__
}

joblib.dump(
    pacote_inferencia,
    pasta_modelos / "preprocessador_rede.joblib"
)

print("Arquivos salvos em:", pasta_modelos.resolve())

In [ ]:
import numpy as np
import joblib
import torch
from torch import nn

# 1. Carrega as configurações e o pré-processamento.
pacote_carregado = joblib.load(
    pasta_modelos / "preprocessador_rede.joblib"
)

camada1, camada2 = pacote_carregado["camadas_ocultas"]

# 2. Reconstrói a estrutura da rede.
rede_carregada = nn.Sequential(
    nn.Linear(pacote_carregado["numero_entradas"], camada1),
    nn.ReLU(),
    nn.Dropout(pacote_carregado["dropout"]),
    nn.Linear(camada1, camada2),
    nn.ReLU(),
    nn.Linear(camada2, 1)
)

# 3. Recupera os pesos aprendidos.
pesos = torch.load(
    pasta_modelos / "rede_neural.pt",
    map_location="cpu",
    weights_only=True
)

rede_carregada.load_state_dict(pesos)
rede_carregada.eval()

# 4. Calcula novamente os scores da validação.
entradas = torch.tensor(
    dados["X"]["validacao"],
    dtype=torch.float32
)

with torch.no_grad():
    scores_recarregados = torch.sigmoid(
        rede_carregada(entradas).squeeze(1)
    ).numpy()

# 5. Compara com os scores que vocês já avaliaram.
np.testing.assert_allclose(
    
    scores_recarregados,
    avaliacao_rede["score"].to_numpy(),
    rtol=1e-5,
    atol=1e-7
)

print("Verificação aprovada: a rede salva reproduz os scores!")

## Verificação da inferência a partir do CSV consolidado

In [ ]:
import sys
import pandas as pd
import numpy as np
import torch

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.tratar_dados_ml import corrigir_distancias, limpar_atributos

# 1. Lê o consolidado, antes do preenchimento dos valores ausentes.
base = pd.read_csv(
    raiz / "Dados" / "pedidos_consolidados_teste.csv",
    dtype={"customer_zip_code_prefix": "string"}
)

# Seleciona os mesmos pedidos da validação, na mesma ordem.
ids = dados["metadados"]["validacao"]["order_id"].tolist()

base = (
    base.set_index("order_id", verify_integrity=True)
    .loc[ids]
    .reset_index()
)

# 2. Aplica as mesmas regras de distância e limpeza.
base = corrigir_distancias(
    base,
    raiz / "Dados" / "Brazilian Logistic",
    registro=[]
)

base = base.set_index("order_id").loc[ids].reset_index()
atributos, _ = limpar_atributos(base)

# 3. Aplica o pré-processador que já foi ajustado no treino.
matriz = pacote_carregado["preprocessador"].transform(
    atributos[pacote_carregado["colunas_entrada"]]
).astype(np.float32)

# Confere se o tratamento reproduziu a matriz original.
np.testing.assert_allclose(
    matriz,
    dados["X"]["validacao"],
    rtol=1e-5,
    atol=1e-7
)

# 4. Executa a rede carregada e confere os scores.
rede_carregada.eval()

with torch.no_grad():
    scores_desde_csv = torch.sigmoid(
        rede_carregada(torch.from_numpy(matriz)).squeeze(1)
    ).numpy()

np.testing.assert_allclose(
    scores_desde_csv,
    avaliacao_rede["score"].to_numpy(),
    rtol=1e-5,
    atol=1e-7
)

print("Conferência aprovada: tratamento e scores reproduzidos!")

## Função de inferência

In [ ]:
def prever_pedidos(pedidos, pacote, modelo):
    """Recebe pedidos consolidados com distâncias já corrigidas."""

    # Limpa e transforma os atributos.
    atributos, _ = limpar_atributos(pedidos)

    matriz = pacote["preprocessador"].transform(
        atributos[pacote["colunas_entrada"]]
    ).astype(np.float32)

    # Calcula os scores usando a rede treinada.
    modelo.eval()

    with torch.no_grad():
        scores = torch.sigmoid(
            modelo(torch.from_numpy(matriz)).squeeze(1)
        ).numpy()

    resultado = pedidos[["order_id"]].copy()
    resultado["score_atraso"] = scores

    return resultado


# Usa os pedidos preparados na conferência anterior.
previsoes = prever_pedidos(base, pacote_carregado, rede_carregada)

# Confere se a função preservou os resultados.
np.testing.assert_allclose(
    previsoes["score_atraso"].to_numpy(),
    scores_desde_csv,
    rtol=1e-5,
    atol=1e-7
)

print("Função de inferência conferida!")
display(previsoes.head())

## Fila de prioridade — 10% dos pedidos

In [ ]:
def priorizar_pedidos(previsoes, fracao):
    resultado = previsoes.copy().reset_index(drop=True)

    quantidade = int(len(resultado) * fracao)
    selecionados = resultado.nlargest(
        quantidade, "score_atraso"
    ).index

    resultado["prioridade"] = 0
    resultado.loc[selecionados, "prioridade"] = 1

    return resultado


previsoes_priorizadas = priorizar_pedidos(
    previsoes,
    pacote_carregado["fracao_fila"]
)

# Confere se reproduz a seleção já avaliada na validação.
np.testing.assert_array_equal(
    previsoes_priorizadas["prioridade"].to_numpy(),
    avaliacao_rede["prioridade"].to_numpy()
)

# Organiza a fila do maior para o menor score.
fila = (
    previsoes_priorizadas[
        previsoes_priorizadas["prioridade"] == 1
    ]
    .sort_values("score_atraso", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print("Priorização conferida!")
print(f"Pedidos avaliados: {len(previsoes_priorizadas)}")
print(f"Pedidos na fila: {len(fila)}")

display(fila.head(10))

## Verificação do módulo de inferência

In [ ]:
from src.inferencia import prever_pedidos, priorizar_pedidos

resultado_modulo = prever_pedidos(
    base, pacote_carregado, rede_carregada
)

resultado_modulo = priorizar_pedidos(
    resultado_modulo,
    pacote_carregado["fracao_fila"]
)

np.testing.assert_allclose(
    resultado_modulo["score_atraso"],
    previsoes_priorizadas["score_atraso"],
    rtol=1e-5,
    atol=1e-7
)

np.testing.assert_array_equal(
    resultado_modulo["prioridade"],
    previsoes_priorizadas["prioridade"]
)

print("Módulo conferido: mesmos scores e mesma fila!")

## Dados de validação para demonstração

In [ ]:
# Seleciona o identificador e os atributos utilizados pelo modelo.
colunas_demo = ["order_id"] + pacote_carregado["colunas_entrada"]

# A variável base já possui as distâncias corrigidas.
entrada_demo = base[colunas_demo].copy()

caminho_demo = raiz / "Dados" / "pedidos_demo_validacao.csv"

entrada_demo.to_csv(caminho_demo, index=False)

print("Arquivo salvo em:", caminho_demo)
print("Pedidos:", len(entrada_demo))
print("Colunas:", len(entrada_demo.columns))

## Conferência dos resultados gerados pelo terminal

In [ ]:
import pandas as pd
import numpy as np

resultado_terminal = pd.read_csv(
    raiz / "Dados" / "previsoes_demo_validacao.csv",
    dtype={"order_id": "string"}
)

# Alinha os resultados pelo identificador e verifica duplicatas.
resultado_terminal = resultado_terminal.set_index(
    "order_id", verify_integrity=True
).sort_index()

referencia_notebook = previsoes_priorizadas.set_index(
    "order_id", verify_integrity=True
).sort_index()

# Confere se são exatamente os mesmos pedidos.
np.testing.assert_array_equal(
    resultado_terminal.index.to_numpy(),
    referencia_notebook.index.to_numpy()
)

# Compara os scores, permitindo pequenas diferenças numéricas.
np.testing.assert_allclose(
    resultado_terminal["score_atraso"],
    referencia_notebook["score_atraso"],
    rtol=1e-5,
    atol=1e-7
)

# A seleção da fila precisa ser exatamente igual.
np.testing.assert_array_equal(
    resultado_terminal["prioridade"],
    referencia_notebook["prioridade"]
)

print("Conferência final aprovada: mesmos pedidos, scores e prioridades!")

## Experimento — redução de atributos

In [ ]:
import numpy as np
from sklearn.feature_selection import VarianceThreshold

# Usa as matrizes originais, preservando a divisão dos pedidos.
X_original_treino = np.asarray(dados["X"]["treino"])
X_original_validacao = np.asarray(dados["X"]["validacao"])

nomes_originais = np.asarray(
    dados["preprocessador"].get_feature_names_out()
)

# Identifica colunas constantes usando SOMENTE o treino.
seletor_constantes = VarianceThreshold(threshold=0)

X_treino_sem_constantes = seletor_constantes.fit_transform(
    X_original_treino
)

# Aplica exatamente a mesma seleção na validação.
X_validacao_sem_constantes = seletor_constantes.transform(
    X_original_validacao
)

colunas_removidas = nomes_originais[
    ~seletor_constantes.get_support()
]

print("Colunas originais:", X_original_treino.shape[1])
print("Colunas removidas:", colunas_removidas.tolist())
print("Treino reduzido:", X_treino_sem_constantes.shape)
print("Validação reduzida:", X_validacao_sem_constantes.shape)

In [ ]:
import pandas as pd
import numpy as np

LIMITE_RARAS = 50

nomes_sem_constantes = nomes_originais[
    seletor_constantes.get_support()
]

# Nomes nas colunas facilitam identificar cada categoria.
treino_agrupado = pd.DataFrame(
    X_treino_sem_constantes,
    columns=nomes_sem_constantes
)

validacao_agrupada = pd.DataFrame(
    X_validacao_sem_constantes,
    columns=nomes_sem_constantes
)

variaveis_categoricas = [
    "customer_state",
    "uf_origem",
    "categoria_principal",
    "tipo_pagamento_principal"
]

grupos_raros = {}
treino_agrupado = treino_agrupado.copy()
validacao_agrupada = validacao_agrupada.copy()


for variavel in variaveis_categoricas:
    prefixo = f"categoricas__{variavel}_"

    colunas = [
        nome for nome in treino_agrupado.columns
        if nome.startswith(prefixo)
    ]

    # Essas colunas são 0/1: a soma conta os pedidos.
    # A frequência é calculada SOMENTE no treino.
    frequencias = treino_agrupado[colunas].sum()
    raras = frequencias[frequencias < LIMITE_RARAS].index.tolist()

    # Só agrupamos quando há pelo menos duas colunas.
    if len(raras) < 2:
        continue

    nova_coluna = f"agrupadas__{variavel}_raras"
    grupos_raros[nova_coluna] = raras

    # Usa exatamente o mesmo agrupamento na validação.
    treino_agrupado[nova_coluna] = treino_agrupado[raras].sum(axis=1)
    validacao_agrupada[nova_coluna] = validacao_agrupada[raras].sum(axis=1)

    treino_agrupado = treino_agrupado.drop(columns=raras)
    validacao_agrupada = validacao_agrupada.drop(columns=raras)

    print(f"{variavel}: {len(raras)} colunas raras agrupadas em 1")

# Matrizes para o próximo treinamento.
X_treino_reduzido = treino_agrupado.to_numpy(dtype=np.float32)
X_validacao_reduzido = validacao_agrupada.to_numpy(dtype=np.float32)
nomes_reduzidos = treino_agrupado.columns.tolist()

assert treino_agrupado.columns.equals(validacao_agrupada.columns)

print("\nTreino reduzido:", X_treino_reduzido.shape)
print("Validação reduzida:", X_validacao_reduzido.shape)
print("Colunas removidas no total:", 159 - len(nomes_reduzidos))

In [ ]:
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

SEMENTE_COMPARACAO = 7
EPOCAS_COMPARACAO = 150
PACIENCIA_COMPARACAO = 50


def avaliar_versao(nome, matriz_treino, matriz_validacao):
    # Mesma semente e configuração para cada versão.
    torch.manual_seed(SEMENTE_COMPARACAO)

    x_t = torch.tensor(
        np.asarray(matriz_treino), dtype=torch.float32
    )
    x_v = torch.tensor(
        np.asarray(matriz_validacao), dtype=torch.float32
    )
    y_t = torch.tensor(
        np.asarray(dados["y"]["treino"]).ravel(),
        dtype=torch.long
    )
    y_v = np.asarray(dados["y"]["validacao"]).ravel().astype(int)

    assert len(x_t) == len(y_t)
    assert len(x_v) == len(y_v)
    assert x_t.shape[1] == x_v.shape[1]

    # Só a quantidade de entradas muda entre as versões.
    rede = nn.Sequential(
        nn.Linear(x_t.shape[1], 128),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 2)
    )

    peso_atraso = (
        (y_t == 0).sum() / (y_t == 1).sum()
    ).item()

    criterio = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, peso_atraso], dtype=torch.float32)
    )

    otimizador = torch.optim.AdamW(
        rede.parameters(), lr=0.0003, weight_decay=0.01
    )

    lotes = DataLoader(
        TensorDataset(x_t, y_t),
        batch_size=256,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEMENTE_COMPARACAO)
    )

    # Reinicia também a sequência aleatória usada pelo dropout.
    torch.manual_seed(SEMENTE_COMPARACAO)

    tamanho_fila = int(len(y_v) * 0.10)
    melhor_quantidade = -1
    sem_melhora = 0

    print(f"\n--- {nome}: {x_t.shape[1]} colunas ---")

    for epoca in range(1, EPOCAS_COMPARACAO + 1):
        rede.train()

        for entradas, alvos in lotes:
            otimizador.zero_grad()
            perda = criterio(rede(entradas), alvos)
            perda.backward()
            otimizador.step()

        rede.eval()
        with torch.no_grad():
            scores = torch.softmax(rede(x_v), dim=1)[:, 1].numpy()

        indices = np.argsort(-scores, kind="stable")[:tamanho_fila]
        encontrados = int(y_v[indices].sum())
        ap = average_precision_score(y_v, scores)

        if encontrados > melhor_quantidade:
            melhor_quantidade = encontrados
            melhor_epoca = epoca
            melhores_scores = scores.copy()
            sem_melhora = 0
        else:
            sem_melhora += 1

        if epoca == 1 or epoca % 5 == 0:
            print(
                f"Época {epoca:03d} | AP: {ap:.4f} | "
                f"Atrasos na fila: {encontrados}"
            )

        if sem_melhora >= PACIENCIA_COMPARACAO:
            print(f"Parada antecipada na época {epoca}.")
            break

    # Avalia os scores da melhor época, não os da última.
    indices = np.argsort(
        -melhores_scores, kind="stable"
    )[:tamanho_fila]

    alertas = np.zeros(len(y_v), dtype=int)
    alertas[indices] = 1

    print(f"Melhor época: {melhor_epoca}")
    print(confusion_matrix(y_v, alertas, labels=[0, 1]))

    resultado = {
        "versao": nome,
        "colunas": x_t.shape[1],
        "melhor_epoca": melhor_epoca,
        "atrasos_top10": melhor_quantidade,
        "precision_%": 100 * precision_score(y_v, alertas, zero_division=0),
        "recall_%": 100 * recall_score(y_v, alertas, zero_division=0),
        "f1_%": 100 * f1_score(y_v, alertas, zero_division=0),
        "AP": average_precision_score(y_v, melhores_scores)
    }

    return resultado, melhores_scores


versoes = {
    "Original": (
        dados["X"]["treino"],
        dados["X"]["validacao"]
    ),
    "Sem constantes": (
        X_treino_sem_constantes,
        X_validacao_sem_constantes
    ),
    "Categorias agrupadas": (
        X_treino_reduzido,
        X_validacao_reduzido
    )
}

resultados_atributos = []
scores_atributos = {}

for nome, (x_t, x_v) in versoes.items():
    resultado, scores = avaliar_versao(nome, x_t, x_v)
    resultados_atributos.append(resultado)
    scores_atributos[nome] = scores

comparacao_atributos = pd.DataFrame(resultados_atributos)
display(comparacao_atributos.round(4))

In [ ]:
# Reaproveita os três resultados anteriores, feitos com semente 7.
resultados_sementes = [
    {**resultado, "semente": 7}
    for resultado in resultados_atributos
]

semente_anterior = SEMENTE_COMPARACAO

try:
    for semente in [42, 123]:
        SEMENTE_COMPARACAO = semente
        print(f"\n===== SEMENTE {semente} =====")

        for nome, (x_t, x_v) in versoes.items():
            resultado, scores = avaliar_versao(nome, x_t, x_v)
            resultado["semente"] = semente
            resultados_sementes.append(resultado)
finally:
    SEMENTE_COMPARACAO = semente_anterior

tabela_sementes = pd.DataFrame(resultados_sementes)

resumo_sementes = (
    tabela_sementes
    .groupby(["versao", "colunas"])
    .agg(
        atrasos_media=("atrasos_top10", "mean"),
        atrasos_desvio=("atrasos_top10", "std"),
        AP_media=("AP", "mean"),
        AP_desvio=("AP", "std")
    )
    .sort_values("atrasos_media", ascending=False)
)

display(tabela_sementes.round(4))
display(resumo_sementes.round(4))

In [ ]:
import numpy as np
import pandas as pd

# Scores da versão original, com semente 7.
scores_diagnostico = np.asarray(
    scores_atributos["Original"]
).ravel()

diagnostico = (
    dados["metadados"]["validacao"][["order_id"]]
    .copy()
    .reset_index(drop=True)
)

diagnostico["atrasou"] = np.asarray(
    dados["y"]["validacao"]
).ravel()

assert len(scores_diagnostico) == len(diagnostico)
assert np.isfinite(scores_diagnostico).all()

diagnostico["score"] = scores_diagnostico

# Mantemos a mesma regra: os 10% maiores scores entram na fila.
quantidade = int(len(diagnostico) * 0.10)
indices = np.argsort(
    -scores_diagnostico, kind="stable"
)[:quantidade]

diagnostico["na_fila"] = False
diagnostico.loc[indices, "na_fila"] = True

atrasou = diagnostico["atrasou"].eq(1)
na_fila = diagnostico["na_fila"]

diagnostico["resultado"] = np.select(
    [
        atrasou & na_fila,
        ~atrasou & na_fila,
        atrasou & ~na_fila,
    ],
    [
        "VP: atrasou e entrou na fila",
        "FP: pontual e entrou na fila",
        "FN: atrasou e ficou fora",
    ],
    default="VN: pontual e ficou fora"
)

# Acrescenta características pelas IDs, evitando misturar pedidos.
colunas = [
    "order_id",
    "customer_state",
    "uf_origem",
    "categoria_principal",
    "prazo_prometido_dias",
    "distancia_max_vendedores_km",
    "faltante_distancia_max_vendedores_km",
]

caracteristicas = pd.read_csv(
    raiz / "Dados" / "pedidos_ml.csv",
    usecols=colunas
)

diagnostico = diagnostico.merge(
    caracteristicas,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert diagnostico["_merge"].eq("both").all()
diagnostico = diagnostico.drop(columns="_merge")

# Distâncias preenchidas pelo tratamento não são medidas reais.
diagnostico.loc[
    diagnostico["faltante_distancia_max_vendedores_km"].eq(1),
    "distancia_max_vendedores_km"
] = np.nan

resumo_erros = diagnostico.groupby("resultado").agg(
    pedidos=("order_id", "size"),
    prazo_medio_dias=("prazo_prometido_dias", "mean"),
    distancia_media_km=("distancia_max_vendedores_km", "mean")
)

display(resumo_erros.round(2))

In [ ]:
from sklearn.decomposition import PCA

# Partimos das 159 entradas originais.
X_base_treino = np.asarray(
    dados["X"]["treino"], dtype=np.float32
)
X_base_validacao = np.asarray(
    dados["X"]["validacao"], dtype=np.float32
)

pca_experimento = PCA(
    n_components=0.95,
    svd_solver="full",
    whiten=False
)

# Aprende a transformação apenas no treino.
X_treino_pca = pca_experimento.fit_transform(
    X_base_treino
).astype(np.float32)

# Aplica exatamente a mesma transformação na validação.
X_validacao_pca = pca_experimento.transform(
    X_base_validacao
).astype(np.float32)

print("Colunas originais:", X_base_treino.shape[1])
print("Componentes após PCA:", pca_experimento.n_components_)
print("Treino:", X_treino_pca.shape)
print("Validação:", X_validacao_pca.shape)
print(
    "Variância preservada:",
    f"{pca_experimento.explained_variance_ratio_.sum():.2%}"
)

In [ ]:
# Confere as configurações usadas na comparação anterior.
assert EPOCAS_COMPARACAO == 150
assert PACIENCIA_COMPARACAO == 50

# Reaproveita os resultados originais das três sementes.
referencia_original = tabela_sementes.loc[
    tabela_sementes["versao"].eq("Original")
].copy()

assert sorted(
    referencia_original["semente"].tolist()
) == [7, 42, 123]

resultados_pca = []
scores_pca = {}

semente_antes_pca = SEMENTE_COMPARACAO

try:
    for semente in [7, 42, 123]:
        SEMENTE_COMPARACAO = semente
        print(f"\n===== PCA — SEMENTE {semente} =====")

        resultado, scores = avaliar_versao(
            "PCA 95%",
            X_treino_pca,
            X_validacao_pca
        )

        resultado["semente"] = semente
        resultados_pca.append(resultado)

        # Guarda os scores de cada execução para análises posteriores.
        scores_pca[semente] = scores.copy()

finally:
    SEMENTE_COMPARACAO = semente_antes_pca

# Compara cada execução com os resultados anteriores.
comparacao_pca = pd.concat(
    [referencia_original, pd.DataFrame(resultados_pca)],
    ignore_index=True
)

display(
    comparacao_pca
    .sort_values(["semente", "versao"])
    .round(4)
)

# Compara as médias e a variação entre sementes.
resumo_pca = (
    comparacao_pca
    .groupby(["versao", "colunas"])
    .agg(
        atrasos_media=("atrasos_top10", "mean"),
        atrasos_desvio=("atrasos_top10", "std"),
        AP_media=("AP", "mean"),
        AP_desvio=("AP", "std")
    )
)

display(resumo_pca.round(4))

## Análise dos erros por distância

Investigar como o desempenho do modelo varia entre faixas de distância vendedor–cliente.

- **Modelo:** rede neural original com 159 entradas, semente 7.
- **Dados:** conjunto de validação.
- **Priorização:** 10% dos pedidos com maiores scores.
- **Indicadores:** quantidade de pedidos, taxa de atraso, proporção na fila, Precision, Recall, falsos positivos e falsos negativos.

Esta análise identifica diferenças entre grupos; não comprova causas dos atrasos.

In [ ]:
# Trabalhamos em uma cópia para preservar o diagnóstico anterior.
analise = diagnostico.copy()

assert analise["order_id"].is_unique

# Identifica os acertos e erros da fila de prioridade.
atrasou = analise["atrasou"].eq(1)
na_fila = analise["na_fila"]

analise["VP"] = atrasou & na_fila
analise["FP"] = ~atrasou & na_fila
analise["FN"] = atrasou & ~na_fila

# Distâncias ausentes ficam em um grupo separado.
distancia = analise["distancia_max_vendedores_km"].copy()

distancia = distancia.mask(
    analise["faltante_distancia_max_vendedores_km"].eq(1)
)

assert distancia.dropna().ge(0).all()

# Faixas para análise; não alteram as previsões.
analise["faixa_distancia"] = pd.cut(
    distancia,
    bins=[0, 100, 500, 1000, 2000, np.inf],
    labels=[
        "Até 100 km",
        "Mais de 100 até 500 km",
        "Mais de 500 até 1.000 km",
        "Mais de 1.000 até 2.000 km",
        "Mais de 2.000 km",
    ],
    include_lowest=True
)

analise["faixa_distancia"] = (
    analise["faixa_distancia"]
    .cat.add_categories(["Sem distância"])
    .fillna("Sem distância")
)

# Conta pedidos, atrasos e erros dentro de cada faixa.
erros_distancia = (
    analise.groupby("faixa_distancia", observed=True)
    .agg(
        pedidos=("order_id", "size"),
        atrasos=("atrasou", "sum"),
        VP=("VP", "sum"),
        FP=("FP", "sum"),
        FN=("FN", "sum")
    )
)

alertas = erros_distancia["VP"] + erros_distancia["FP"]

erros_distancia["taxa_atraso_%"] = (
    100 * erros_distancia["atrasos"] / erros_distancia["pedidos"]
)

erros_distancia["na_fila_%"] = (
    100 * alertas / erros_distancia["pedidos"]
)

erros_distancia["precision_%"] = (
    100 * erros_distancia["VP"] / alertas.replace(0, np.nan)
)

erros_distancia["recall_%"] = (
    100 * erros_distancia["VP"]
    / erros_distancia["atrasos"].replace(0, np.nan)
)

assert erros_distancia["pedidos"].sum() == len(analise)

print(f"Taxa geral de atrasos: {analise['atrasou'].mean():.2%}")
display(erros_distancia.round(2))

## Análise dos erros por prazo prometido

Comparar o desempenho da rede neural original, semente 7, entre
faixas de prazo prometido no conjunto de validação.

Examinar a frequência real de atrasos, a proporção de pedidos
priorizados, Precision, Recall e falsos positivos e negativos.

As diferenças observadas são associações, não evidências de causalidade.

In [ ]:
# Usa a tabela "analise" criada na etapa anterior.
prazo = analise["prazo_prometido_dias"]

assert prazo.dropna().ge(0).all(), "Existem prazos negativos; confira os dados."

# Define faixas para investigação.
analise["faixa_prazo"] = pd.cut(
    prazo,
    bins=[0, 7, 14, 21, 30, np.inf],
    labels=[
        "Até 7 dias",
        "Mais de 7 até 14 dias",
        "Mais de 14 até 21 dias",
        "Mais de 21 até 30 dias",
        "Mais de 30 dias",
    ],
    include_lowest=True
)

analise["faixa_prazo"] = (
    analise["faixa_prazo"]
    .cat.add_categories(["Sem prazo"])
    .fillna("Sem prazo")
)

# Conta os resultados dentro de cada faixa.
erros_prazo = (
    analise.groupby("faixa_prazo", observed=True)
    .agg(
        pedidos=("order_id", "size"),
        atrasos=("atrasou", "sum"),
        VP=("VP", "sum"),
        FP=("FP", "sum"),
        FN=("FN", "sum")
    )
)

alertas_prazo = erros_prazo["VP"] + erros_prazo["FP"]

erros_prazo["taxa_atraso_%"] = (
    100 * erros_prazo["atrasos"] / erros_prazo["pedidos"]
)

erros_prazo["na_fila_%"] = (
    100 * alertas_prazo / erros_prazo["pedidos"]
)

erros_prazo["precision_%"] = (
    100 * erros_prazo["VP"]
    / alertas_prazo.replace(0, np.nan)
)

erros_prazo["recall_%"] = (
    100 * erros_prazo["VP"]
    / erros_prazo["atrasos"].replace(0, np.nan)
)

# Confere se todos os pedidos foram contabilizados.
assert erros_prazo["pedidos"].sum() == len(analise)

print(f"Pedidos analisados: {len(analise)}")
print(f"Taxa geral de atrasos: {analise['atrasou'].mean():.2%}")

display(erros_prazo.round(2))

## Análise dos erros por prazo e distância

Investigar o desempenho da rede neural original, semente 7,
combinando faixas de prazo prometido e distância vendedor–cliente.

A análise utiliza o conjunto de validação e a mesma fila de 10%.
Serão consideradas as quantidades de pedidos e atrasos, Precision,
Recall, falsos positivos e falsos negativos.

Grupos pequenos serão interpretados com cautela.

In [ ]:
# Usa as faixas e os indicadores criados nas etapas anteriores.
erros_cruzados = (
    analise.groupby(
        ["faixa_prazo", "faixa_distancia"],
        observed=True
    )
    .agg(
        pedidos=("order_id", "size"),
        atrasos=("atrasou", "sum"),
        VP=("VP", "sum"),
        FP=("FP", "sum"),
        FN=("FN", "sum")
    )
)

alertas_cruzados = erros_cruzados["VP"] + erros_cruzados["FP"]

# Frequência real de atraso em cada combinação.
erros_cruzados["taxa_atraso_%"] = (
    100 * erros_cruzados["atrasos"] / erros_cruzados["pedidos"]
)

# Parcela de cada grupo que entrou na fila.
erros_cruzados["na_fila_%"] = (
    100 * alertas_cruzados / erros_cruzados["pedidos"]
)

# Entre os priorizados, quantos realmente atrasaram?
erros_cruzados["precision_%"] = (
    100 * erros_cruzados["VP"]
    / alertas_cruzados.replace(0, np.nan)
)

# Dos atrasos reais, quantos entraram na fila?
erros_cruzados["recall_%"] = (
    100 * erros_cruzados["VP"]
    / erros_cruzados["atrasos"].replace(0, np.nan)
)

# Confere se o agrupamento preservou os totais.
assert erros_cruzados["pedidos"].sum() == len(analise)
assert erros_cruzados["atrasos"].sum() == analise["atrasou"].sum()

# Mostra primeiro os grupos com mais atrasos não encontrados.
tabela_cruzada = (
    erros_cruzados
    .reset_index()
    .sort_values("FN", ascending=False)
    .reset_index(drop=True)
)

print("Combinações com pedidos:", len(tabela_cruzada))

with pd.option_context("display.max_rows", None):
    display(tabela_cruzada.round(2))

## Análise dos erros por estado e categoria

Comparar o desempenho da rede neural original, semente 7,
por estado de destino e categoria principal do produto.

Examinar a validação completa, as três combinações de prazo
e distância identificadas anteriormente e o restante da validação.

Os recortes foram selecionados após observar os erros.
Esta análise é exploratória e serve para formular hipóteses.

In [ ]:
# Seleciona exatamente as três combinações já identificadas.
grupos_investigados = (
    (
        analise["faixa_prazo"].eq("Mais de 21 até 30 dias")
        & analise["faixa_distancia"].isin([
            "Mais de 100 até 500 km",
            "Mais de 500 até 1.000 km",
        ])
    )
    |
    (
        analise["faixa_prazo"].eq("Mais de 14 até 21 dias")
        & analise["faixa_distancia"].eq("Mais de 100 até 500 km")
    )
)

recortes = {
    "Validação completa": analise,
    "Três grupos investigados": analise.loc[grupos_investigados],
    "Restante da validação": analise.loc[~grupos_investigados],
}

for nome, base in recortes.items():
    print(
        f"{nome}: {len(base)} pedidos | "
        f"{base['atrasou'].sum()} atrasos | "
        f"{base['FN'].sum()} falsos negativos"
    )

In [ ]:
def analisar_segmentos(coluna):
    resultados = []

    for nome, base in recortes.items():
        tabela = (
            base.groupby(coluna, observed=True, dropna=False)
            .agg(
                pedidos=("order_id", "size"),
                atrasos=("atrasou", "sum"),
                VP=("VP", "sum"),
                FP=("FP", "sum"),
                FN=("FN", "sum")
            )
        )

        alertas = tabela["VP"] + tabela["FP"]

        tabela["taxa_atraso_%"] = (
            100 * tabela["atrasos"] / tabela["pedidos"]
        )

        tabela["na_fila_%"] = (
            100 * alertas / tabela["pedidos"]
        )

        tabela["precision_%"] = (
            100 * tabela["VP"] / alertas.replace(0, np.nan)
        )

        tabela["recall_%"] = (
            100 * tabela["VP"]
            / tabela["atrasos"].replace(0, np.nan)
        )

        # Confere se todos os pedidos do recorte foram contabilizados.
        assert tabela["pedidos"].sum() == len(base)

        tabela = (
            tabela.reset_index()
            .sort_values("FN", ascending=False)
        )

        tabela.insert(0, "recorte", nome)
        resultados.append(tabela)

    return pd.concat(resultados, ignore_index=True)

In [ ]:
# customer_state representa o estado de destino.
erros_estado = analisar_segmentos("customer_state")

with pd.option_context("display.max_rows", None):
    display(erros_estado.round(2))

In [ ]:
erros_categoria = analisar_segmentos("categoria_principal")

with pd.option_context("display.max_rows", None):
    display(erros_categoria.round(2))

## Experimento: inclusão da localização do cliente

Adicionar latitude e longitude aproximadas do cliente às 159 entradas
originais, usando a mediana das coordenadas por prefixo de CEP.

A imputação e a padronização das novas entradas serão ajustadas
somente no treino.

Comparar posteriormente o desempenho com as sementes 7, 42 e 123,
mantendo a configuração da rede e a fila de 10%.

In [ ]:
import numpy as np
import pandas as pd

# Lê somente as informações geográficas necessárias.
geo = pd.read_csv(
    raiz / "Dados" / "Brazilian Logistic" / "olist_geolocation_dataset.csv",
    usecols=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
    ],
    dtype={"geolocation_zip_code_prefix": "string"}
)

geo["geolocation_zip_code_prefix"] = (
    geo["geolocation_zip_code_prefix"].str.zfill(5)
)

# Mesmo filtro geográfico usado no tratamento das distâncias.
plausiveis = (
    geo["geolocation_lat"].between(-34, 6)
    & geo["geolocation_lng"].between(-74, -28)
)

# Uma localização aproximada por prefixo de CEP.
coordenadas_cep = (
    geo.loc[plausiveis]
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        latitude_cliente=("geolocation_lat", "median"),
        longitude_cliente=("geolocation_lng", "median")
    )
    .rename(columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix"
    })
)

# Recupera o CEP de cada pedido do consolidado existente.
pedidos_cep = pd.read_csv(
    raiz / "Dados" / "pedidos_consolidados_teste.csv",
    usecols=["order_id", "customer_zip_code_prefix"],
    dtype={"customer_zip_code_prefix": "string"}
)

pedidos_cep["customer_zip_code_prefix"] = (
    pedidos_cep["customer_zip_code_prefix"].str.zfill(5)
)

assert pedidos_cep["order_id"].notna().all()
assert pedidos_cep["order_id"].is_unique

coordenadas_pedidos = pedidos_cep.merge(
    coordenadas_cep,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

In [ ]:
def obter_coordenadas(particao):
    ids = (
        dados["metadados"][particao][["order_id"]]
        .reset_index(drop=True)
    )

    resultado = ids.merge(
        coordenadas_pedidos,
        on="order_id",
        how="left",
        sort=False,
        validate="one_to_one",
        indicator=True
    )

    # Confere que todos os IDs foram encontrados e mantiveram a ordem.
    assert resultado["_merge"].eq("both").all()
    assert resultado["order_id"].equals(ids["order_id"])
    assert len(resultado) == len(dados["X"][particao])

    return resultado[["latitude_cliente", "longitude_cliente"]].copy()


coordenadas_treino = obter_coordenadas("treino")
coordenadas_validacao = obter_coordenadas("validacao")

print("Ausências no treino:")
print(coordenadas_treino.isna().sum())

print("\nAusências na validação:")
print(coordenadas_validacao.isna().sum())

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

assert coordenadas_treino.notna().any().all()

preprocessador_geo = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# Aprende medianas e escalas apenas no treino.
geo_treino_tratado = preprocessador_geo.fit_transform(
    coordenadas_treino
)

# Aplica o mesmo tratamento à validação.
geo_validacao_tratada = preprocessador_geo.transform(
    coordenadas_validacao
)

X_treino_geo = np.column_stack([
    dados["X"]["treino"],
    geo_treino_tratado
]).astype(np.float32)

X_validacao_geo = np.column_stack([
    dados["X"]["validacao"],
    geo_validacao_tratada
]).astype(np.float32)

assert np.isfinite(X_treino_geo).all()
assert np.isfinite(X_validacao_geo).all()

print("Treino com coordenadas:", X_treino_geo.shape)
print("Validação com coordenadas:", X_validacao_geo.shape)

## Comparação: modelo original versus inclusão de coordenadas

Comparar as versões com 159 e 161 entradas nas sementes 7, 42 e 123.

Manter a mesma configuração de treinamento e selecionar a melhor época
pela quantidade de atrasos encontrados na fila de 10% da validação.

Avaliar a média de atrasos encontrados, Average Precision
e a variação entre sementes.

In [ ]:
# Confere as configurações usadas no experimento original.
assert EPOCAS_COMPARACAO == 150
assert PACIENCIA_COMPARACAO == 50

assert X_treino_geo.shape[1] == 161
assert X_validacao_geo.shape[1] == 161

# Reaproveita os resultados originais das três sementes.
referencia_original_geo = tabela_sementes.loc[
    tabela_sementes["versao"].eq("Original")
].copy()

assert sorted(
    referencia_original_geo["semente"].tolist()
) == [7, 42, 123]

resultados_geo = []
scores_geo = {}

semente_antes_geo = SEMENTE_COMPARACAO

try:
    for semente in [7, 42, 123]:
        SEMENTE_COMPARACAO = semente
        print(f"\n===== COORDENADAS — SEMENTE {semente} =====")

        resultado, scores = avaliar_versao(
            "Original + coordenadas",
            X_treino_geo,
            X_validacao_geo
        )

        resultado["semente"] = semente
        resultados_geo.append(resultado)

        # Guarda os scores de cada semente para análise posterior.
        scores_geo[semente] = scores.copy()

finally:
    SEMENTE_COMPARACAO = semente_antes_geo

# Resultados individuais das duas versões.
comparacao_geo = pd.concat(
    [referencia_original_geo, pd.DataFrame(resultados_geo)],
    ignore_index=True
)

display(
    comparacao_geo
    .sort_values(["semente", "versao"])
    .round(4)
)

# Médias e variação entre as três sementes.
resumo_geo = (
    comparacao_geo
    .groupby(["versao", "colunas"])
    .agg(
        atrasos_media=("atrasos_top10", "mean"),
        atrasos_desvio=("atrasos_top10", "std"),
        AP_media=("AP", "mean"),
        AP_desvio=("AP", "std")
    )
)

display(resumo_geo.round(4))

## Comparação dos dados de treino e validação por segmento

Comparar estados de destino e categorias principais quanto a:
- Quantidade de pedidos e atrasos.
- Frequência de atraso.
- Participação no total de pedidos de cada partição.

O objetivo é investigar diferenças entre os conjuntos que possam
ajudar a explicar as limitações do modelo.

As taxas da validação serão usadas exclusivamente para diagnóstico.

In [ ]:
# Recupera apenas as características necessárias.
segmentos = pd.read_csv(
    raiz / "Dados" / "pedidos_ml.csv",
    usecols=["order_id", "customer_state", "categoria_principal"]
)

assert segmentos["order_id"].notna().all()
assert segmentos["order_id"].is_unique

bases_comparacao = {}

for particao in ["treino", "validacao"]:
    ids = (
        dados["metadados"][particao][["order_id"]]
        .reset_index(drop=True)
    )

    base = ids.merge(
        segmentos,
        on="order_id",
        how="left",
        sort=False,
        validate="one_to_one",
        indicator=True
    )

    assert base["_merge"].eq("both").all()
    assert base["order_id"].equals(ids["order_id"])

    alvos = np.asarray(dados["y"][particao]).ravel()
    assert len(base) == len(alvos)

    base["atrasou"] = alvos
    bases_comparacao[particao] = base.drop(columns="_merge")

    print(
        f"{particao}: {len(base)} pedidos | "
        f"{int(base['atrasou'].sum())} atrasos | "
        f"Taxa: {base['atrasou'].mean():.2%}"
    )

In [ ]:
def comparar_particoes(coluna):
    tabelas = []

    for particao, base in bases_comparacao.items():
        tabela = (
            base.groupby(coluna, observed=True, dropna=False)
            .agg(
                pedidos=("order_id", "size"),
                atrasos=("atrasou", "sum")
            )
        )

        tabela["taxa_%"] = (
            100 * tabela["atrasos"] / tabela["pedidos"]
        )

        tabela["participacao_%"] = (
            100 * tabela["pedidos"] / len(base)
        )

        assert tabela["pedidos"].sum() == len(base)

        # Identifica a origem das colunas: treino ou validação.
        tabelas.append(tabela.add_prefix(f"{particao}_"))

    # Alinha os mesmos estados ou categorias lado a lado.
    comparacao = pd.concat(tabelas, axis=1)

    for particao in ["treino", "validacao"]:
        contagens = [
            f"{particao}_pedidos",
            f"{particao}_atrasos"
        ]

        comparacao[contagens] = (
            comparacao[contagens].fillna(0).astype(int)
        )

        participacao = f"{particao}_participacao_%"
        comparacao[participacao] = (
            comparacao[participacao].fillna(0)
        )

    # Taxas de grupos sem pedidos permanecem NaN.
    comparacao["variacao_taxa_pp"] = (
        comparacao["validacao_taxa_%"]
        - comparacao["treino_taxa_%"]
    )

    comparacao["variacao_participacao_pp"] = (
        comparacao["validacao_participacao_%"]
        - comparacao["treino_participacao_%"]
    )

    return (
        comparacao.reset_index()
        .sort_values("validacao_atrasos", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
comparacao_estados = comparar_particoes("customer_state")

with pd.option_context("display.max_rows", None):
    display(comparacao_estados.round(2))

In [ ]:
comparacao_categorias = comparar_particoes("categoria_principal")

with pd.option_context("display.max_rows", None):
    display(comparacao_categorias.round(2))

## Evolução mensal dos atrasos e auditoria dos cortes temporais

Comparar, por mês de aprovação, a frequência de atraso antes e
depois das exclusões temporais aplicadas ao treino e à validação.

“Antes” corresponde aos pedidos que passaram pelas regras de qualidade,
mas ainda não passaram pela exclusão temporal.

Desfechos conhecidos após os cortes são consultados apenas nesta
auditoria retrospectiva, sem acrescentá-los ao treinamento.

In [ ]:
import sys

# Permite importar as funções do projeto pelo notebook.
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.tratar_dados_ml import selecionar_coorte

corte_validacao = pd.Timestamp(dados["cortes"]["validacao"])
corte_teste = pd.Timestamp(dados["cortes"]["teste"])

entrada_auditoria = pd.read_csv(
    raiz / "Dados" / "pedidos_consolidados_teste.csv"
)

# Considera somente aprovações anteriores ao período de teste.
aprovacoes = pd.to_datetime(
    entrada_auditoria["order_approved_at"], errors="coerce"
)

entrada_auditoria = entrada_auditoria.loc[
    aprovacoes.lt(corte_teste)
].copy()

# Reutiliza as mesmas regras de qualidade e definição dos alvos.
auditoria = selecionar_coorte(entrada_auditoria, [])

auditoria["particao"] = np.where(
    auditoria["order_approved_at"].lt(corte_validacao),
    "treino",
    "validacao"
)

auditoria["limite"] = auditoria["particao"].map({
    "treino": corte_validacao,
    "validacao": corte_teste
})

# Reproduz a exclusão aplicada pelo pipeline.
auditoria["excluido"] = (
    auditoria["rotulo_disponivel_em"] >= auditoria["limite"]
)

auditoria["mes_aprovacao"] = (
    auditoria["order_approved_at"].dt.to_period("M").astype(str)
)

auditoria["atrasos_excluidos"] = (
    auditoria["atrasou"] * auditoria["excluido"]
)

# Confere a correspondência com os conjuntos usados no modelo.
for particao in ["treino", "validacao"]:
    usados = auditoria.loc[
        auditoria["particao"].eq(particao)
        & ~auditoria["excluido"]
    ].set_index("order_id")

    ids = dados["metadados"][particao]["order_id"]

    assert usados.index.is_unique
    assert set(usados.index) == set(ids)
    assert np.array_equal(
        usados.loc[ids, "atrasou"].to_numpy(),
        np.asarray(dados["y"][particao]).ravel()
    )

    print(f"{particao}: {len(usados)} pedidos e alvos conferidos.")

In [ ]:
def resumir_exclusoes(colunas):
    tabela = (
        auditoria.groupby(colunas, observed=True)
        .agg(
            pedidos_antes=("order_id", "size"),
            atrasos_antes=("atrasou", "sum"),
            pedidos_excluidos=("excluido", "sum"),
            atrasos_excluidos=("atrasos_excluidos", "sum")
        )
    )

    tabela["pedidos_usados"] = (
        tabela["pedidos_antes"] - tabela["pedidos_excluidos"]
    )

    tabela["atrasos_usados"] = (
        tabela["atrasos_antes"] - tabela["atrasos_excluidos"]
    )

    # Calcula a taxa em cada população, com seu próprio denominador.
    for grupo in ["antes", "usados", "excluidos"]:
        tabela[f"taxa_{grupo}_%"] = (
            100 * tabela[f"atrasos_{grupo}"]
            / tabela[f"pedidos_{grupo}"].replace(0, np.nan)
        )

    tabela["excluidos_%"] = (
        100 * tabela["pedidos_excluidos"]
        / tabela["pedidos_antes"]
    )

    return tabela.reset_index()

In [ ]:
resumo_cortes = resumir_exclusoes(["particao"])
display(resumo_cortes.round(2))

In [ ]:
resumo_mensal = resumir_exclusoes(
    ["particao", "mes_aprovacao"]
)

with pd.option_context("display.max_rows", None):
    display(resumo_mensal.round(2))